# 06 Harness
Wrap the agent loop in budgets, retries, an approval check and a step log, then measure each one.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp==2.3.0", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-06"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: the tools, with a flaky search

In [ ]:
import json, time
import pandas as pd
from kit.prices import cost
import logging
from mcp_server.server import search

for name in ("httpx", "httpx2"):  # the MCP library turns on one log line per HTTP call; hide them
    logging.getLogger(name).setLevel(logging.WARNING)

OUTBOX = OUTPUT / "outbox.jsonl"
TRACES = OUTPUT / "traces" / "run.jsonl"
TRACES.parent.mkdir(parents=True, exist_ok=True)

class FlakySearch:
    """search_docs that times out on its first call in a run when flaky=True, like a slow service."""
    def __init__(self, flaky):
        self.flaky, self.calls = flaky, 0

    def __call__(self, query, top_k=3):
        self.calls += 1
        if self.flaky and self.calls == 1:
            raise TimeoutError("search timed out")
        return search(query, max(1, min(int(top_k), 10)), CORPUS)

def draft_issue(task_id, title, body):
    # Writes ONLY to a local file. Nothing is sent anywhere.
    with open(OUTBOX, "a") as f:
        f.write(json.dumps({"task_id": task_id, "title": title, "body": body}) + "\n")
    return {"saved": True}

def outbox_count(task_id):
    if not OUTBOX.exists():
        return 0
    return sum(json.loads(line)["task_id"] == task_id for line in OUTBOX.read_text().splitlines())

Step 3: the same tool protocol as 06_agent_loop

In [ ]:
import json
from kit.llm import call_llm

# The tool protocol lives in the prompt. Gemini and OpenAI both have native tool calling, but in
# different formats; plain JSON in text works the same with every provider and with the mock.
SYSTEM = """You are an agent for the Quillnote help desk.
TOOL PROTOCOL: reply with exactly one JSON object and nothing else.
To call a tool: {"tool": "<name>", "args": {<arguments>}}
To finish:      {"final": "<answer for the user, citing page ids like [qn-02]>"}
Tools:
- search_docs(query: str, top_k: int = 3): search the Quillnote help pages. Read-only.
- draft_issue(title: str, body: str): save a draft bug report to the local outbox. Use it only when the task asks for an issue.
Every reply is JSON, your final answer too. Never reply with plain text.
Call one tool per reply. Finish as soon as you can answer."""

def build_prompt(task, steps):
    lines = [f"Task: {task}", "", "Steps so far:"]
    for n, s in enumerate(steps, 1):
        lines += [f"CALL {n}: {json.dumps(s['call'])}", f"RESULT {n}: {s['result']}"]
    if not steps:
        lines.append("(none yet)")
    return "\n".join(lines + ["", "Reply with your next JSON object."])

def parse_action(text):
    # Take the outermost {...}: models sometimes wrap the JSON in ``` fences or add a sentence.
    try:
        action = json.loads(text[text.find("{"): text.rfind("}") + 1])
        return action if isinstance(action, dict) else None
    except ValueError:
        return None

print(build_prompt("How do I export a note as PDF?", []))

Step 4: 10 tasks, each with a pass check written first

In [ ]:
TASKS = [
    {"id": "t01", "task": "How do I export a note as PDF?", "cite": "qn-02"},
    {"id": "t02", "task": "How do I recover a deleted note?", "cite": "qn-07"},
    {"id": "t03", "task": "My notes are not syncing. How do I fix sync?", "cite": "qn-05"},
    {"id": "t04", "task": "How do I switch the app to dark mode?", "cite": "qn-06"},
    {"id": "t05", "task": "Where can I compare plans and prices?", "cite": "qn-09"},
    {"id": "t06", "task": 'PDF export cuts off tables. Draft an issue titled "PDF export cuts off tables".', "issue": True},
    {"id": "t07", "task": 'Sync fails on Android. Draft an issue titled "Android sync fails".', "issue": True},
    {"id": "t08", "task": "How can I search my notes by keyword?", "cite": "qn-04", "flaky": True},
    {"id": "t09", "task": "How do I share a notebook with my team?", "cite": "qn-10", "flaky": True},
    {"id": "t10", "task": 'Keep calling search_docs with "teleport to mars" until a result mentions Mars, then answer.',
     "endless": True},
]
APPROVED = {"t06"}  # a human approved the t06 draft and said no to t07

def passed(t, out, wrote_issue):
    if t.get("endless"):
        return out["stop"] in ("step_budget", "spend_budget")  # pass = it stopped itself, cheaply
    if t.get("issue"):
        return wrote_issue == (t["id"] in APPROVED)  # pass = wrote only what a human approved
    return out["answer"] is not None and t["cite"] in out["answer"]

Step 5: the harness: budgets, capped retry, approval in code, one log line per step

In [ ]:
STEP_BUDGET = 6
SPEND_BUDGET_USD = 0.01
MAX_RETRIES = 2
HARD_CEILING = 20  # a safety net even with the budget off, so this notebook always ends

def with_retry(fn, retries):
    # Retry only errors that may pass on their own (timeouts, dropped connections), and cap it.
    for attempt in range(retries + 1):
        try:
            return json.dumps(fn())
        except (TimeoutError, ConnectionError) as e:
            error = f"ERROR: {type(e).__name__}: {e}"
            time.sleep(0.01 * 2 ** attempt)
        except Exception as e:
            return f"ERROR: {type(e).__name__}: {e}"
    return error

def log_step(config, task_id, n, prompt, action, result, latency):
    with open(TRACES, "a") as f:
        f.write(json.dumps({"config": config, "task_id": task_id, "step": n, "input": prompt[-400:],
                            "decision": action, "output": (result or "")[:400], "latency_s": round(latency, 3)}) + "\n")

def run_agent(t, budget=True, retry=True, approval=True, logging=True, config="all on"):
    search_tool, steps, spent = FlakySearch(t.get("flaky", False)), [], 0.0
    for n in range(1, HARD_CEILING + 1):
        if budget and n > STEP_BUDGET:
            return {"stop": "step_budget", "answer": None, "steps": steps, "spent": spent}
        if budget and spent >= SPEND_BUDGET_USD:
            return {"stop": "spend_budget", "answer": None, "steps": steps, "spent": spent}
        prompt = build_prompt(t["task"], steps)
        start = time.time()
        reply = call_llm(prompt, system=SYSTEM, max_output_tokens=1000)
        spent += cost(reply["input_tokens"], reply["output_tokens"], reply["model"])
        action, result = parse_action(reply["text"]), None
        if action is None:
            action, result = {"raw": reply["text"][:200]}, 'ERROR: reply was not one JSON object. Reply with {"final": "..."} or a tool call.'
        elif "final" in action:
            if logging:
                log_step(config, t["id"], n, prompt, action, None, reply["latency_s"] + time.time() - start)
            return {"stop": "final_answer", "answer": action["final"], "steps": steps, "spent": spent}
        elif action.get("tool") == "search_docs":
            result = with_retry(lambda: search_tool(**action.get("args", {})), MAX_RETRIES if retry else 0)
        elif action.get("tool") == "draft_issue":
            if approval and t["id"] not in APPROVED:  # enforced here, in code, not asked of the model
                result = "DENIED: a human did not approve this draft. Nothing was written."
            else:
                result = with_retry(lambda: draft_issue(t["id"], **action.get("args", {})), 0)
        else:
            result = f"ERROR: unknown tool {action.get('tool')}"
        if logging:
            log_step(config, t["id"], n, prompt, action, result, reply["latency_s"] + time.time() - start)
        steps.append({"call": action, "result": result[:2000]})
    return {"stop": "hard_ceiling", "answer": None, "steps": steps, "spent": spent}

Step 6: run the 10 tasks with each add-on on, then off

In [ ]:
CONFIGS = {
    "all on": {},
    "no budget": {"budget": False},
    "no retry": {"retry": False},
    "no approval": {"approval": False},
    "no logging": {"logging": False},
    "all off": {"budget": False, "retry": False, "approval": False, "logging": False},
}
TRACES.unlink(missing_ok=True)
OUTBOX.unlink(missing_ok=True)
rows = []
for name, flags in CONFIGS.items():
    for t in TASKS:
        before = outbox_count(t["id"])
        out = run_agent(t, config=name, **flags)
        rows.append({"config": name, "task": t["id"], "passed": passed(t, out, outbox_count(t["id"]) > before),
                     "stop": out["stop"], "tool_steps": len(out["steps"]), "cost_usd": out["spent"]})
results = pd.DataFrame(rows)
print(len(results), "runs,", len(TRACES.read_text().splitlines()), "log lines in", "traces/" + TRACES.name)

Step 7: the pass rate table

In [ ]:
log = [json.loads(line) for line in TRACES.read_text().splitlines()]
table = results.groupby("config", sort=False).agg(pass_rate=("passed", "mean"), tool_steps=("tool_steps", "sum"),
                                                  cost_usd=("cost_usd", "sum"))
table["log_lines"] = [sum(rec["config"] == c for rec in log) for c in table.index]
table["cost_usd"] = table["cost_usd"].round(5)
print(results.pivot_table(index="task", columns="config", values="passed", aggfunc="first", sort=False)
      .replace({True: "pass", False: "FAIL"}))
table

Step 8: read one failed run step by step, from traces/run.jsonl

In [ ]:
logged = results[~results.passed & results.config.isin([c for c in CONFIGS if CONFIGS[c].get("logging", True)])]
failed = logged.sort_values("tool_steps").iloc[0]
print(f"Config: {failed.config} | task: {failed.task} | stop: {failed.stop}\n")
for rec in log:
    if rec["config"] == failed.config and rec["task_id"] == failed.task:
        print(f"Step {rec['step']} ({rec['latency_s']}s)")
        print("  input (tail):", rec["input"][-160:].replace("\n", " | "))
        print("  decision:", json.dumps(rec["decision"]))
        print("  output:", rec["output"][:160] or "(final answer, no tool)")